# 02 – Preprocessing and labelling (corrected)

This notebook replaces `02_preprocessing_feature_engineering.ipynb`. It builds one feature table and a fixed, seeded train/validation/test split that all later notebooks share.

**What changed compared with the thesis version**

1. **Connection-level labels instead of file-level labels.** The thesis labelled every connection in the `normal_DDoS_1` capture as an attack. That capture also contains normal background traffic. The provided `GroundTruth_Network_*.csv` files only cover port scanning (23–24 April 2019), not this DDoS capture, so they cannot label it. Instead, a connection is labelled as an attack when it originates from the TON_IoT attacker hosts `192.168.1.30`–`192.168.1.39` (the same hosts that appear as sources in the scanning ground truth). **This is an assumption and is stated as such.**
2. **The timestamp is no longer a model feature.** The normal capture is from 2 April 2019 and the DDoS capture from 25 April 2019, so `ts` alone separates the two classes. Source and destination ports are also dropped, because ephemeral port numbers are noise when treated as numbers.
3. **Log-scaling of byte, packet and duration counts**, which are heavily skewed.
4. **`history` is reduced to its 20 most common values plus `other`**, instead of one column per distinct value (the original produced ~770 columns, mostly from `history`).
5. **Scaling is no longer done here.** Each model notebook fits its scaler on training data only, so no test information leaks into scaling.
6. **One fixed, seeded split**, saved in the `split` column: 80% train / 20% test (stratified), with 20% of the training normal traffic held out as `val_normal` for choosing alert thresholds.

In [1]:
import os, random, json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)
from sklearn.model_selection import train_test_split

In [2]:
COLS = ["ts", "uid", "id.orig_h", "id.orig_p", "id.resp_h", "id.resp_p", "proto", "service",
        "duration", "orig_bytes", "resp_bytes", "conn_state", "local_orig", "local_resp",
        "missed_bytes", "history", "orig_pkts", "orig_ip_bytes", "resp_pkts", "resp_ip_bytes",
        "tunnel_parents"]

normal = pd.read_csv("../../data/normal_2/conn.csv", header=None, names=COLS)
ddos_capture = pd.read_csv("../../data/normal_DDoS_1/conn.csv", header=None, names=COLS)

for name, d in [("normal_2", normal), ("normal_DDoS_1", ddos_capture)]:
    t = pd.to_datetime(d["ts"], unit="s")
    print(f"{name}: {len(d):,} connections, {t.min()} to {t.max()}")

normal_2: 7,179 connections, 2019-04-02 18:35:27 to 2019-04-02 21:36:41
normal_DDoS_1: 123,536 connections, 2019-04-25 14:48:47 to 2019-04-25 15:10:14


## Labelling
Normal capture: all connections are normal (0). DDoS capture: attack (1) only when the connection originates from an attacker host.

In [3]:
ATTACKER_HOSTS = {f"192.168.1.{i}" for i in range(30, 40)}

normal["label"] = 0
ddos_capture["label"] = ddos_capture["id.orig_h"].isin(ATTACKER_HOSTS).astype(int)
normal["capture"] = "normal_2"
ddos_capture["capture"] = "normal_DDoS_1"

raw = pd.concat([normal, ddos_capture], ignore_index=True)
print(pd.crosstab(raw["capture"], raw["label"], margins=True))
print("\nTop source hosts in the DDoS capture:")
print(ddos_capture["id.orig_h"].value_counts().head(6))

label              0       1     All
capture                             
normal_2        7179       0    7179
normal_DDoS_1  12224  111312  123536
All            19403  111312  130715

Top source hosts in the DDoS capture:
id.orig_h
192.168.1.30     57702
192.168.1.31     53401
192.168.1.190     1450
192.168.1.180      882
172.17.0.3         732
192.168.1.38       107
Name: count, dtype: int64


## Features

In [4]:
NUMERIC = ["duration", "orig_bytes", "resp_bytes", "missed_bytes",
           "orig_pkts", "orig_ip_bytes", "resp_pkts", "resp_ip_bytes"]
CATEGORICAL = ["proto", "service", "conn_state", "history_grp"]

feat = raw[NUMERIC + ["proto", "service", "conn_state", "history"]].copy()
for c in NUMERIC:
    v = pd.to_numeric(feat[c].replace("-", np.nan), errors="coerce").fillna(0).clip(lower=0)
    feat[c] = np.log1p(v)

top_history = feat["history"].value_counts().head(20).index
feat["history_grp"] = np.where(feat["history"].isin(top_history), feat["history"], "other")
feat = feat.drop(columns=["history"])

features = pd.get_dummies(feat, columns=CATEGORICAL, dtype="float32")
features[NUMERIC] = features[NUMERIC].astype("float32")
print("Feature matrix:", features.shape)
print(features.columns.tolist())

Feature matrix: (130715, 50)
['duration', 'orig_bytes', 'resp_bytes', 'missed_bytes', 'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes', 'proto_icmp', 'proto_tcp', 'proto_udp', 'service_-', 'service_dhcp', 'service_dns', 'service_http', 'service_ssl', 'conn_state_OTH', 'conn_state_REJ', 'conn_state_RSTO', 'conn_state_RSTOS0', 'conn_state_RSTR', 'conn_state_RSTRH', 'conn_state_S0', 'conn_state_S1', 'conn_state_S2', 'conn_state_S3', 'conn_state_SF', 'conn_state_SH', 'conn_state_SHR', 'history_grp_A', 'history_grp_CcC', 'history_grp_Cd', 'history_grp_D', 'history_grp_Dc', 'history_grp_Dd', 'history_grp_F', 'history_grp_S', 'history_grp_Sh', 'history_grp_ShA', 'history_grp_ShADad', 'history_grp_ShADadFf', 'history_grp_ShADadf', 'history_grp_ShADdFaf', 'history_grp_ShAf', 'history_grp_^d', 'history_grp_^f', 'history_grp_^fA', 'history_grp_^h', 'history_grp_^hA', 'history_grp_other']


## Fixed split (shared by all later notebooks)

In [5]:
idx = np.arange(len(features))
idx_train, idx_test = train_test_split(idx, test_size=0.2, stratify=raw["label"], random_state=SEED)

train_normal = idx_train[raw["label"].values[idx_train] == 0]
_, idx_val_normal = train_test_split(train_normal, test_size=0.2, random_state=SEED)

split = np.array(["train"] * len(features), dtype=object)
split[idx_test] = "test"
split[idx_val_normal] = "val_normal"

out = features.copy()
out["label"] = raw["label"].values
out["capture"] = raw["capture"].values
out["split"] = split
out.to_csv("features_conn.csv", index=False)

print(pd.crosstab(out["split"], out["label"]))
print("Saved features_conn.csv")

label           0      1
split                   
test         3881  22262
train       12417  89050
val_normal   3105      0
Saved features_conn.csv
